# Road Damage Object Detection — YOLO Comparison

**Goal:** train, compare, tune and deploy YOLO models that detect road damage.

**Pipeline**
1. Setup & download the dataset from Roboflow (6 classes, 4930 / 1146 / 681 images)
2. Train 3 YOLO models with identical settings: **YOLOv8s, YOLO11s, YOLOv10s**
3. Pick the best architecture on the **validation** set
4. Hyperparameter tuning on that architecture
5. Evaluate every model on the **test** set: mAP, Precision, Recall, F1, IoU, FPS, model size, ONNX/CPU deployment efficiency
6. Comparison table + charts
7. Export the best model 

## 1. Setup

In [4]:
!pip install -q ultralytics roboflow onnx onnxruntime onnxslim ultralytics-thop

import os
os.environ["WANDB_MODE"] = "disabled"

from ultralytics import settings
settings.update({"wandb": False})

from kaggle_secrets import UserSecretsClient
api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")

## 2. Configuration
Change `WORKSPACE` and `PROJECT` to the values from Roboflow → *Download Dataset → YOLOv8 → Show download code*.

In [5]:
# ---- Roboflow ----
WORKSPACE = "samaas-workspace"      
PROJECT   = "road-damage-l1ju7-puubt"        
VERSION   = 1

# ---- Training (same for every model, so the comparison is fair) ----
EPOCHS   = 60
PATIENCE = 15
IMGSZ    = 640
BATCH    = 32          # 16 per GPU on T4 x2
DEVICE   = [0, 1]
WORKERS  = 4
SEED     = 0

WORK    = "/kaggle/working"
RUNS    = f"{WORK}/runs"
RESULTS = f"{WORK}/results"
FIGS    = f"{RESULTS}/figures"
os.makedirs(FIGS, exist_ok=True)

BASE_MODELS = {
    "yolov8s":  "yolov8s.pt",
    "yolo11s":  "yolo11s.pt",
    "yolov10s": "yolov10s.pt",
}

# Baseline hyperparameters, written explicitly so "auto" does not pick
# a different optimizer behind our back
BASE_HP = dict(optimizer="SGD", lr0=0.01, momentum=0.937, weight_decay=0.0005)

## 3. Download dataset & fix `data.yaml` paths

In [6]:
import yaml
from roboflow import Roboflow

os.chdir(WORK)
rf = Roboflow(api_key=api_key)
ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

yaml_path = f"{ds.location}/data.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

cfg["path"]  = ds.location
cfg["train"] = "train/images"
cfg["val"]   = "valid/images"
cfg["test"]  = "test/images"
with open(yaml_path, "w") as f:
    yaml.safe_dump(cfg, f)

CLASS_NAMES = cfg["names"]
print(cfg["nc"], CLASS_NAMES)
for split in ["train", "valid", "test"]:
    print(split, len(os.listdir(f"{ds.location}/{split}/images")))

loading Roboflow workspace...
loading Roboflow project...
6 ['alligator', 'block', 'crack', 'longitudinal', 'pothole', 'transverse']
train 4930
valid 1146
test 681


### Dataset: class distribution (instances per split)

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

def count_instances(split):
    counts = {c: 0 for c in CLASS_NAMES}
    for lbl in glob.glob(f"{ds.location}/{split}/labels/*.txt"):
        with open(lbl) as f:
            for line in f:
                if line.strip():
                    counts[CLASS_NAMES[int(float(line.split()[0]))]] += 1
    return counts

dist = pd.DataFrame({s: count_instances(s) for s in ["train", "valid", "test"]})
dist["total"] = dist.sum(axis=1)
dist = dist.sort_values("total", ascending=False)
display(dist)
dist.to_csv(f"{RESULTS}/class_distribution.csv")

dist[["train", "valid", "test"]].plot(kind="bar", stacked=True, figsize=(8, 4))
plt.title("Instances per class"); plt.ylabel("boxes"); plt.xticks(rotation=30)
plt.tight_layout(); plt.savefig(f"{FIGS}/class_distribution.png", dpi=150); plt.show()

## 4. Training helper

In [ ]:
import gc, json, time, torch
from ultralytics import YOLO

TIME_LOG = f"{RESULTS}/train_time_minutes.json"
train_time = json.load(open(TIME_LOG)) if os.path.exists(TIME_LOG) else {}

def best_pt(name):
    return f"{RUNS}/{name}/weights/best.pt"

def train_run(name, weights, **hp):
    if os.path.exists(best_pt(name)) and name in train_time:
        print(f"[skip] {name} already trained")
        return
    print(f"\n========== Training {name} ==========")
    t0 = time.time()
    YOLO(weights).train(
        data=yaml_path, epochs=EPOCHS, patience=PATIENCE,
        imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=WORKERS, seed=SEED,
        project=RUNS, name=name, exist_ok=True, plots=True,
        **hp,
    )
    train_time[name] = round((time.time() - t0) / 60, 1)
    json.dump(train_time, open(TIME_LOG, "w"), indent=2)
    gc.collect(); torch.cuda.empty_cache()

## 5. Train the 3 baseline models (identical settings)

In [ ]:
for name, weights in BASE_MODELS.items():
    train_run(name, weights, **BASE_HP)
print(train_time)

## 6. Select the best architecture on the **validation** set
The test set is kept untouched for the final comparison.

In [ ]:
def validate(name, split):
    return YOLO(best_pt(name)).val(
        data=yaml_path, split=split, imgsz=IMGSZ, batch=32, device=0,
        plots=(split == "test"), verbose=False,
        project=f"{RUNS}/_eval", name=f"{name}_{split}", exist_ok=True,
    )

val_scores = {n: validate(n, "val").box.map for n in BASE_MODELS}
best_arch = max(val_scores, key=val_scores.get)
print("val mAP50-95:", {k: round(v, 4) for k, v in val_scores.items()})
print("Best architecture:", best_arch)

## 7. Hyperparameter tuning on the best architecture

Baseline = SGD, lr0 = 0.01, default augmentation. Two tuned variants:

| Run | Change | Why |
|---|---|---|
| `tune_adamw` | AdamW, lr0 = 0.001 | adaptive optimizer often converges faster on mid-size datasets |
| `tune_cos_aug` | SGD + cosine LR, mixup 0.1, scale 0.7, close_mosaic 5 | stronger augmentation for the small classes (block, transverse) + smoother LR decay |

(`model.tune()` with a genetic search would need dozens of full trainings, which is too long for a Kaggle session, so a targeted grid is used.)

In [ ]:
TUNE_CONFIGS = {
    "tune_adamw":   dict(optimizer="AdamW", lr0=0.001, weight_decay=0.0005),
    "tune_cos_aug": dict(BASE_HP, cos_lr=True, mixup=0.1, scale=0.7, close_mosaic=5),
}

for tname, hp in TUNE_CONFIGS.items():
    train_run(f"{best_arch}_{tname}", BASE_MODELS[best_arch], **hp)
print(train_time)

## 8. Evaluation helpers

* **mAP50, mAP50-95, Precision, Recall** → `model.val(split="test")`
* **Mean IoU** → for every test image, predictions (conf ≥ 0.25) are matched one-to-one with ground-truth boxes of the same class (greedy, highest IoU first, IoU ≥ 0.5). Mean IoU = average IoU of the matched pairs, i.e. how tight the boxes are for correct detections.
* **FPS** → end-to-end `predict()` on single images (read + preprocess + inference + NMS), batch = 1, after warm-up. Measured on GPU (T4) and CPU.
* **Model size** → `.pt` file size, parameters, GFLOPs
* **Deployment efficiency** → export to ONNX, measure ONNX file size and CPU FPS with ONNX Runtime

In [ ]:
import numpy as np
from ultralytics.utils.metrics import box_iou

TEST_IMGS = sorted(glob.glob(f"{ds.location}/test/images/*"))
TEST_LBLS = f"{ds.location}/test/labels"

def load_gt(img_path, w, h):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    path = os.path.join(TEST_LBLS, stem + ".txt")
    cls, boxes = [], []
    if os.path.exists(path):
        for line in open(path):
            v = line.split()
            if len(v) < 5:
                continue
            c, nums = int(float(v[0])), np.array(v[1:], dtype=float)
            if len(nums) == 4:                       # xc yc w h
                xc, yc, bw, bh = nums
                x1, y1, x2, y2 = xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2
            else:                                    # polygon -> bbox
                xs, ys = nums[0::2], nums[1::2]
                x1, y1, x2, y2 = xs.min(), ys.min(), xs.max(), ys.max()
            cls.append(c); boxes.append([x1 * w, y1 * h, x2 * w, y2 * h])
    return np.array(cls, dtype=int), np.array(boxes, dtype=np.float32).reshape(-1, 4)

def mean_iou(model, conf=0.25, iou_thr=0.5, bs=16):
    ious = []
    for s in range(0, len(TEST_IMGS), bs):                     # 16 images at a time
        for r in model.predict(TEST_IMGS[s:s + bs], conf=conf, imgsz=IMGSZ,
                               device=0, verbose=False):
            h, w = r.orig_shape
            g_cls, g_box = load_gt(r.path, w, h)
            if len(g_box) == 0 or len(r.boxes) == 0:
                continue
            p_box = r.boxes.xyxy.cpu()
            p_cls = r.boxes.cls.cpu().numpy().astype(int)
            M = box_iou(p_box, torch.from_numpy(g_box)).numpy()
            M[p_cls[:, None] != g_cls[None, :]] = 0
            while M.size:
                i, j = np.unravel_index(M.argmax(), M.shape)
                if M[i, j] < iou_thr:
                    break
                ious.append(M[i, j]); M[i, :] = 0; M[:, j] = 0
    return float(np.mean(ious)) if ious else 0.0

def measure_fps(weights, device, n=200, warmup=10):
    m = YOLO(weights, task="detect")
    imgs = TEST_IMGS[:n]
    for p in imgs[:warmup]:
        m.predict(p, imgsz=IMGSZ, device=device, verbose=False)
    t0 = time.perf_counter()
    for p in imgs:
        m.predict(p, imgsz=IMGSZ, device=device, verbose=False)
    dt = time.perf_counter() - t0
    return len(imgs) / dt, 1000 * dt / len(imgs)

def model_complexity(weights):
    from ultralytics.utils.torch_utils import get_flops
    m = YOLO(weights)
    params = sum(p.numel() for p in m.model.parameters())
    try:
        gflops = get_flops(m.model, IMGSZ) or float("nan")
    except Exception:
        gflops = float("nan")
    return params / 1e6, gflops

def size_mb(path):
    return os.path.getsize(path) / 1e6

## 9. Evaluate all models on the test set

In [ ]:
ALL_RUNS = list(BASE_MODELS) + [f"{best_arch}_{t}" for t in TUNE_CONFIGS]
rows, per_class = [], []

for name in ALL_RUNS:
    print(f"\n---------- Evaluating {name} ----------")
    pt = best_pt(name)
    vm = validate(name, "val")
    tm = validate(name, "test")
    P, R = tm.box.mp, tm.box.mr

    gpu_fps, gpu_ms = measure_fps(pt, device=0, n=200)
    cpu_fps, cpu_ms = measure_fps(pt, device="cpu", n=50)

    onnx_path = YOLO(pt).export(format="onnx", imgsz=IMGSZ, simplify=True, device="cpu")
    onnx_fps, onnx_ms = measure_fps(onnx_path, device="cpu", n=50)

    params_m, gflops = model_complexity(pt)

    rows.append({
        "model": name,
        "val_mAP50_95": vm.box.map,
        "test_mAP50": tm.box.map50,
        "test_mAP50_95": tm.box.map,
        "precision": P,
        "recall": R,
        "F1": 2 * P * R / (P + R + 1e-9),
        "mean_IoU": mean_iou(YOLO(pt)),
        "GPU_FPS": gpu_fps, "GPU_ms": gpu_ms,
        "CPU_FPS_pytorch": cpu_fps,
        "CPU_FPS_onnx": onnx_fps, "CPU_ms_onnx": onnx_ms,
        "pt_size_MB": size_mb(pt),
        "onnx_size_MB": size_mb(onnx_path),
        "params_M": params_m,
        "GFLOPs": gflops,
        "train_min": train_time.get(name),
    })

    for k, ci in enumerate(tm.box.ap_class_index):
        per_class.append({
            "model": name, "class": CLASS_NAMES[int(ci)],
            "precision": tm.box.p[k], "recall": tm.box.r[k],
            "AP50": tm.box.ap50[k], "AP50_95": tm.box.ap[k],
        })
    del vm, tm
    gc.collect(); torch.cuda.empty_cache()
    

results = pd.DataFrame(rows).set_index("model")
per_class_df = pd.DataFrame(per_class)
results.to_csv(f"{RESULTS}/comparison.csv")
per_class_df.to_csv(f"{RESULTS}/per_class.csv", index=False)
display(results.round(4))

### Per-class AP50 (test)

In [ ]:
ap50_table = per_class_df.pivot(index="class", columns="model", values="AP50")
display(ap50_table.round(3))
ap50_table.to_csv(f"{RESULTS}/per_class_AP50.csv")

ap50_table.plot(kind="bar", figsize=(10, 4))
plt.title("Per-class AP50 on test set"); plt.ylabel("AP50"); plt.xticks(rotation=30)
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{FIGS}/per_class_ap50.png", dpi=150); plt.show()

## 10. Comparison charts

In [ ]:
def bar(col, title, ylabel, fname):
    ax = results[col].plot(kind="bar", figsize=(8, 4), color="#4C72B0")
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3g}", (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=8)
    plt.title(title); plt.ylabel(ylabel); plt.xticks(rotation=20)
    plt.tight_layout(); plt.savefig(f"{FIGS}/{fname}", dpi=150); plt.show()

bar("test_mAP50_95", "Test mAP50-95", "mAP50-95", "map50_95.png")
bar("test_mAP50",    "Test mAP50",    "mAP50",    "map50.png")
bar("GPU_FPS",       "Inference speed (T4 GPU, batch 1)", "FPS", "fps_gpu.png")
bar("CPU_FPS_onnx",  "Inference speed (CPU, ONNX Runtime)", "FPS", "fps_cpu_onnx.png")
bar("pt_size_MB",    "Model size (.pt)", "MB", "model_size.png")

# precision / recall / F1 / IoU grouped
results[["precision", "recall", "F1", "mean_IoU"]].plot(kind="bar", figsize=(9, 4))
plt.title("Precision, Recall, F1, Mean IoU (test)"); plt.xticks(rotation=20)
plt.ylim(0, 1); plt.tight_layout(); plt.savefig(f"{FIGS}/pr_f1_iou.png", dpi=150); plt.show()

# accuracy vs speed trade-off
plt.figure(figsize=(7, 5))
for name, r in results.iterrows():
    plt.scatter(r["GPU_FPS"], r["test_mAP50_95"], s=r["pt_size_MB"] * 10, alpha=0.6)
    plt.annotate(name, (r["GPU_FPS"], r["test_mAP50_95"]), fontsize=8,
                 xytext=(5, 5), textcoords="offset points")
plt.xlabel("GPU FPS"); plt.ylabel("test mAP50-95")
plt.title("Accuracy vs speed (bubble = model size)")
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f"{FIGS}/accuracy_vs_speed.png", dpi=150); plt.show()

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name in ALL_RUNS:
    df = pd.read_csv(f"{RUNS}/{name}/results.csv")
    df.columns = df.columns.str.strip()
    axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label=name)
    axes[1].plot(df["epoch"], df["metrics/mAP50(B)"], label=name)
    axes[2].plot(df["epoch"], df["val/box_loss"], label=name)
for ax, t in zip(axes, ["val mAP50-95", "val mAP50", "val box loss"]):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIGS}/training_curves.png", dpi=150); plt.show()

## 11. Best model → export for deployment
Selected by **validation** mAP50-95 (not test) to avoid overfitting the model choice to the test set.

In [ ]:
import shutil

best_name = results["val_mAP50_95"].idxmax()
print("Best model:", best_name)
display(results.loc[[best_name]].round(4).T)

shutil.copy(best_pt(best_name), f"{RESULTS}/best_model.pt")
shutil.copy(best_pt(best_name).replace(".pt", ".onnx"), f"{RESULTS}/best_model.onnx")
with open(f"{RESULTS}/best_model.txt", "w") as f:
    f.write(best_name)

# copy confusion matrix of the best model
cm = f"{RUNS}/_eval/{best_name}_test/confusion_matrix_normalized.png"
if os.path.exists(cm):
    shutil.copy(cm, f"{FIGS}/confusion_matrix_best.png")

### Sample predictions (best model, test images)

In [ ]:
import random
random.seed(1)
best = YOLO(f"{RESULTS}/best_model.pt")
samples = random.sample(TEST_IMGS, 8)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, p in zip(axes.ravel(), samples):
    r = best.predict(p, conf=0.25, imgsz=IMGSZ, device=0, verbose=False)[0]
    ax.imshow(r.plot()[:, :, ::-1]); ax.axis("off")
    ax.set_title(os.path.basename(p)[:25], fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIGS}/sample_predictions.png", dpi=120); plt.show()

## 12. Zip results for download
`results.zip` contains: comparison tables (CSV), all figures, `best_model.pt`, `best_model.onnx`.

In [ ]:
shutil.make_archive(f"{WORK}/results", "zip", RESULTS)
print(os.listdir(RESULTS))
print("results.zip:", round(size_mb(f"{WORK}/results.zip"), 1), "MB")

## 13. Results & Discussion

### Final comparison (60 epochs, test set = 681 images)

| Model | mAP50 | mAP50-95 | Precision | Recall | F1 | Mean IoU | GPU FPS | CPU FPS (ONNX) | Size (MB) | Params (M) | GFLOPs |
|---|---|---|---|---|---|---|---|---|---|---|---|
| YOLOv8s | 0.597 | 0.370 | 0.637 | 0.545 | 0.588 | 0.767 | 53.6 | 6.2 | 22.5 | 11.1 | 28.7 |
| YOLO11s | 0.597 | 0.372 | 0.626 | 0.552 | 0.587 | 0.779 | 60.7 | 7.1 | 19.2 | 9.4 | **21.7** |
| YOLOv10s | 0.593 | 0.369 | 0.587 | **0.575** | 0.581 | 0.778 | 63.2 | **7.4** | **16.5** | **8.1** | 24.9 |
| **YOLOv10s + AdamW** | **0.606** | **0.372** | **0.659** | 0.545 | **0.597** | 0.775 | **63.6** | **7.4** | **16.5** | **8.1** | 24.9 |
| YOLOv10s + cos LR & aug | 0.582 | 0.357 | 0.600 | 0.539 | 0.568 | 0.782 | 63.4 | 7.3 | 16.5 | 8.1 | 24.9 |

### Architecture comparison
With identical training (60 epochs, SGD, same seed), the three architectures are practically tied: validation mAP50-95 is 0.284 / 0.283 / 0.284 and test mAP50-95 is 0.370 / 0.372 / 0.369. Differences of ±0.003 are within run-to-run noise, so accuracy alone does not separate them. **Efficiency does**: YOLOv10s has 27% fewer parameters than YOLOv8s, a 27% smaller file, and is 18% faster on GPU (63.2 vs 53.6 FPS) and 19% faster on CPU with ONNX. Interestingly, YOLO11s has the lowest GFLOPs (21.7) but YOLOv10s is still faster. YOLOv10's NMS-free head removes the post-processing step, showing that real latency depends on more than FLOPs.

### Hyperparameter tuning (on YOLOv10s, selected on validation)
- **AdamW (lr0 = 0.001)** gave the best model overall: highest validation mAP50-95 (0.287), test mAP50 (0.606), mAP50-95 (0.372), precision (0.659) and F1 (0.597). At 30 epochs AdamW was slightly *worse* than SGD, so it benefits from the longer schedule.
- **Cosine LR + stronger augmentation** (mixup 0.1, scale 0.7) improved the most on validation between 30 and 60 epochs (+0.018) but remained the weakest on test. Stronger augmentation makes training harder and needs even more epochs to pay off.

### Effect of training length (30 vs 60 epochs)
Doubling the epochs improved validation mAP50-95 for every model (+0.002 to +0.018), but test results were mixed: YOLO11s and both tuned models improved, while YOLOv8s and YOLOv10s dropped slightly (−0.005). Training time doubled (≈21–26 → 40–51 min per model). The validation curves flatten after ~50 epochs, so 60 epochs is close to the plateau for this dataset at 640 px. Further gains are more likely from data and resolution changes than from longer training.

### Precision vs recall
The tuned AdamW model is the most precise (fewer false alarms), while the plain YOLOv10s has the highest recall (0.575, fewer missed defects). For a road-inspection tool where a human reviews detections, either trade-off is acceptable; the confidence slider in the app lets the user move along this curve.

### Per-class behaviour (best model, AP50)
- **Strong:** crack 0.83, block 0.83, pothole 0.70, transverse 0.67 (the best transverse score of all models).
- **Weak:** alligator 0.35 and longitudinal 0.26, even though they are the 2nd and 3rd most frequent classes. Class imbalance is therefore **not** the main cause.
- The confusion matrix shows almost no confusion *between* damage types. Errors are with **background**: 43% of background false positives are predicted as alligator, and 10% of longitudinal cracks are missed. Alligator cracking covers large regions with unclear borders, and longitudinal cracks are long and thin, so box extents are ambiguous and annotations are likely inconsistent. This hurts mAP50-95 even more than mAP50.
- Mean IoU of correct detections is ~0.77–0.78 for every model, so localisation quality is similar across architectures.

### Validation vs test gap
Test mAP50-95 (~0.37) is consistently higher than validation (~0.28). The test split has a different class mix (fewer hard alligator/longitudinal boxes relative to its size), so models should only be compared within the same split. This is also why the best model was chosen on validation, not test.

### Deployment
**Selected model: YOLOv10s + AdamW.** It is the most accurate model while also being the smallest (16.5 MB, 8.1 M params) and fastest (63.6 FPS on a T4 GPU). It runs in real time on GPU; on CPU it reaches ~7.4 FPS with ONNX Runtime, which suits image-based detection and slower video processing. The ONNX export is 29 MB and gives a ~25% CPU speed-up over PyTorch.

### Limitations & future work
- Train at a higher resolution (800–1024 px) to help thin longitudinal/transverse cracks.
- Review annotation consistency for alligator and longitudinal cracks.
- Repeat runs with several seeds to confirm whether the small differences between architectures are significant.
- Try a medium model (YOLOv10m) or INT8/FP16 export for faster CPU and edge deployment.